# Biohydrogenfabrikken

## Stoffstrømmer, reaktorkinetikk og hydrogenproduksjon

### Pilotprosjekt for Matematikk 1, logistikk og bioteknologi

Et biologisk reaktorsystem omdanner karbohydratrikt avløpsvann til organiske syrer og hydrogen. Prosessen har to hovedtrinn:

```text
karbohydrat
   ├──> laktat ──> acetat
   │            ├─> butyrat
   │            └─> hydrogen
   ├──> acetat
   └──> hydrogen
```

Den første mikrobielle gruppen omdanner karbohydrat raskt. En annen gruppe omdanner laktat langsommere. Derfor kan laktat bygge seg opp før det senere omdannes videre.

Prosjektet har fem deler:

1. **Reaksjonsnettverk:** støkiometrisk matrise og samlet produktutbytte
2. **Én biologisk reaksjon:** skalar ODE og metningskinetikk
3. **Batchreaktor:** et femkomponents vektor-ODE-system
4. **Kontinuerlig reaktor:** innstrømning, utstrømning og oppholdstid
5. **Egenmoder:** linearisering, stabilitet og biologiske tidsskalaer

### Læringsmål

Etter prosjektet skal du kunne

- bygge en støkiometrisk matrise fra et reaksjonsdiagram,
- tolke fortegn, kolonner og produktutbytter,
- bruke matriser til å kombinere flere reaksjonstrinn,
- implementere Monod-lignende metningskinetikk,
- bruke Euler på en skalar ODE,
- skrive en reaksjonsmodell som $\dot x=Nr(x)$,
- simulere karbohydrat, laktat, acetat, butyrat og hydrogen,
- formulere en kontinuerlig omrørt reaktor,
- finne en stasjonær tilstand numerisk,
- beregne en numerisk Jacobimatrise,
- tolke egenverdier som raske og langsomme biologiske moder.

### Viktig avgrensning

Utbyttekoeffisientene er oppgitt på COD-basis. De skal ikke tolkes som vanlige massefraksjoner. Modellen beskriver ikke pH, gass-væske-likevekt, temperaturvariasjon, alle mikrobielle grupper eller alle mellomprodukter. Parameterne brukes som et pedagogisk referansetilfelle.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Referanseparametre

Vi bruker to reaksjoner:

$$C\longrightarrow f_{C,L}L+f_{C,A}A+f_{C,H}H,$$

$$L\longrightarrow f_{L,A}A+f_{L,B}B+f_{L,H}H.$$

Her betegner

- $C$: karbohydrat
- $L$: laktat
- $A$: acetat
- $B$: butyrat
- $H$: kumulativt produsert hydrogen, uttrykt på modellens COD-basis

Utbyttekoeffisienter:

$$f_{C,L}=0.72,\quad f_{C,A}=0.26,\quad f_{C,H}=0.02,$$

$$f_{L,A}=0.19,\quad f_{L,B}=1.12,\quad f_{L,H}=0.06.$$

Kinetiske parametre:

$$k_C=3.62\ \mathrm{h^{-1}},\quad K_{S,C}=76\ \mathrm{g\,COD/L},$$

$$k_L=0.030\ \mathrm{h^{-1}},\quad K_{S,L}=0.82\ \mathrm{g\,COD/L}.$$

In [ ]:
f_CL, f_CA, f_CH = 0.72, 0.26, 0.02
f_LA, f_LB, f_LH = 0.19, 1.12, 0.06

k_C = 3.62      # 1/h
KS_C = 76.0     # g COD/L
k_L = 0.030     # 1/h
KS_L = 0.82     # g COD/L

# Effektive, konstante biomassefaktorer i hovedmodellen.
X_C = 8.0       # g COD/L, pedagogisk scenarioverdi
X_L = 3.0       # g COD/L, pedagogisk scenarioverdi

# Del A: Reaksjonsnettverket som lineær algebra

## A.1 Støkiometrisk matrise

Tilstandsvektoren er

$$
\boxed{x=(C,L,A,B,H)^T.}
$$

Reaksjonsratevektoren er

$$
\boxed{r=(r_C,r_L)^T.}
$$

Da kan modellen skrives

$$
\boxed{\dot x=Nr,}
$$

med

$$
N=
\begin{pmatrix}
-1&0\\
f_{C,L}&-1\\
f_{C,A}&f_{L,A}\\
0&f_{L,B}\\
f_{C,H}&f_{L,H}
\end{pmatrix}.
$$

## Oppgave A1: Bygg og tolk $N$

Bygg matrisen og forklar hver kolonne. Beregn $Nr$ når

$$r_C=4.0,\qquad r_L=1.0.$$

In [ ]:
N = np.array([
    [..., ...],
    [..., ...],
    [..., ...],
    [..., ...],
    [..., ...]
], dtype=float)

r_test = np.array([4.0, 1.0])
dx_test = ...

print("N =
", N)
print("Stoffendring N @ r =", dx_test)
print("Rang:", ...)

## A.2 Samlet utbytte gjennom to trinn

Anta at én enhet karbohydrat først reagerer, og at alt produsert laktat deretter reagerer videre.

Direkte produkter fra karbohydratruten:

$$
y_C=
\begin{pmatrix}
f_{C,A}\\0\\f_{C,H}
\end{pmatrix}.
$$

Produkter fra én enhet laktat:

$$
y_L=
\begin{pmatrix}
f_{L,A}\\f_{L,B}\\f_{L,H}
\end{pmatrix}.
$$

Samlet utbytte blir

$$
\boxed{y_{tot}=y_C+f_{C,L}y_L.}
$$

In [ ]:
y_C = np.array([f_CA, 0.0, f_CH])
y_L = np.array([f_LA, f_LB, f_LH])
y_total = ...

print("Samlet [acetat, butyrat, hydrogen] =", y_total)

## Oppgave A2: Delvis laktatomdanning

La $\theta\in[0,1]$ være andelen produsert laktat som reagerer videre:

$$
\boxed{y(\theta)=y_C+\theta f_{C,L}y_L.}
$$

Plott acetat-, butyrat- og hydrogenutbytte som funksjon av $\theta$. Hvor stor andel av hydrogenet kommer fra andre reaksjonstrinn når $\theta=1$?

In [ ]:
theta = np.linspace(0.0, 1.0, 101)
Y_theta = ...

# Plott de tre produktutbyttene.

## A.3 COD-basis og modellregnskap

At $f_{L,B}=1.12$ er større enn én betyr ikke uten videre at fysisk masse skapes. Koeffisientene er formulert på COD-basis, altså oksidasjonsekvivalenter.

Diskuter:

- hvorfor COD ikke er det samme som våt masse
- hvorfor koeffisientene ikke bør summeres som vanlige massefraksjoner
- hvorfor en full modell også trenger elektroner, gassfase og flere stoffbalanser

# Del B: Karbohydratreaksjonen som skalar ODE

## B.1 Metningskinetikk

Med konstant effektiv biomasse bruker vi

$$
\boxed{r_C(C)=k_CX_C\frac{C}{K_{S,C}+C}.}
$$

Karbohydratbalansen i en batch er

$$
\boxed{\dot C=-r_C(C).}
$$

For $C\ll K_{S,C}$ er reaksjonen omtrent første orden. For $C\gg K_{S,C}$ er den omtrent nullte orden.

In [ ]:
def rate_C(C):
    C_pos = np.maximum(C, 0.0)
    return k_C*X_C*C_pos/(KS_C+C_pos)


def C_ode(t, C):
    return -rate_C(C)

## Oppgave B1: Studer ratefunksjonen

Plott $r_C(C)$ for $0\leq C\leq400$ g COD/L. Marker $K_{S,C}$ og maksimalraten $k_CX_C$.

In [ ]:
C_grid = np.linspace(0.0, 400.0, 500)
r_grid = ...

# Plott og marker KS_C.

## B.2 Euler

Bruk startkonsentrasjonen

$$C(0)=160\ \mathrm{g\,COD/L}.$$

Simuler 48 timer. Hindre at et numerisk steg gir negativ konsentrasjon.

In [ ]:
def euler_skalar(f, y0, sluttid, dt):
    n = int(round(sluttid/dt))
    t = np.linspace(0.0, n*dt, n+1)
    y = np.zeros(n+1)
    y[0] = y0
    for j in range(n):
        y[j+1] = ...
    return t, y

# Sammenlign dt = 0.5, 0.1 og 0.02 timer.

## Oppgave B2: Halveringstid og parameterfølsomhet

Finn numerisk tidspunktet der $C$ første gang blir mindre enn $C(0)/2$. Sammenlign:

- halv og dobbel $X_C$
- halv og dobbel $k_C$
- liten og stor $K_{S,C}$
- ulike startkonsentrasjoner

# Del C: Femkomponents batchmodell

## C.1 Reaksjonsratene

Laktatraten er

$$
\boxed{r_L(L)=k_LX_L\frac{L}{K_{S,L}+L}.}
$$

Hele batchmodellen er

$$
\boxed{\dot x=Nr(x).}
$$

Komponentvis:

$$\dot C=-r_C,$$

$$\dot L=f_{C,L}r_C-r_L,$$

$$\dot A=f_{C,A}r_C+f_{L,A}r_L,$$

$$\dot B=f_{L,B}r_L,$$

$$\dot H=f_{C,H}r_C+f_{L,H}r_L.$$

In [ ]:
def rate_L(L):
    L_pos = np.maximum(L, 0.0)
    return k_L*X_L*L_pos/(KS_L+L_pos)


def batch_ode(t, x):
    C, L, A, B, H = x
    r = np.array([rate_C(C), rate_L(L)])
    return ...

## Oppgave C1: Implementer vektor-Euler

Start med

$$x(0)=(160,0,0,0,0)^T$$

og simuler 240 timer.

In [ ]:
def euler_system(f, x0, sluttid, dt):
    n = int(round(sluttid/dt))
    t = np.linspace(0.0, n*dt, n+1)
    X = np.zeros((n+1, len(x0)))
    X[0] = x0
    for j in range(n):
        X[j+1] = ...
        X[j+1] = np.maximum(X[j+1], 0.0)
    return t, X

x0_batch = np.array([160.0, 0.0, 0.0, 0.0, 0.0])

# Simuler og plott alle fem komponenter.

## Oppgave C2: Toppunkt for laktat

Finn:

- tidspunktet for maksimal laktatkonsentrasjon
- maksimal laktatkonsentrasjon
- tidspunktet der 95 prosent av karbohydratet er brukt
- hydrogenproduksjonen ved disse tidspunktene

Forklar toppen i $L(t)$ ved å sammenligne produksjonsleddet $f_{C,L}r_C$ med forbruksleddet $r_L$.

## Oppgave C3: Hydrogen fra første og andre trinn

Utvid tilstanden med to bokføringsvariable:

- $H_C$: hydrogen fra karbohydratruten
- $H_L$: hydrogen fra laktatruten

slik at

$$\dot H_C=f_{C,H}r_C,$$

$$\dot H_L=f_{L,H}r_L.$$

Kontroller at

$$H=H_C+H_L.$$

In [ ]:
# Lag en utvidet batchmodell med seks eller sju relevante bokføringsvariable.

## C.2 To alternative råstoff

Sammenlign batcher med samme start-COD:

1. alt som karbohydrat
2. alt som laktat
3. 75 prosent karbohydrat og 25 prosent laktat

Sammenlign hydrogenprofil, produktfordeling og nødvendig prosesstid.

# Del D: Kontinuerlig omrørt reaktor

## D.1 Fortynning og innløp

Med hydraulisk oppholdstid $\tau$ blir modellen

$$
\boxed{
\dot x=Nr(x)+\frac1\tau(x_{inn}-x).}
$$

Vi bruker innløp med karbohydrat, men uten produktene:

$$
x_{inn}=(C_{inn},0,0,0,0)^T.
$$

In [ ]:
C_inn = 160.0
x_inn = np.array([C_inn, 0.0, 0.0, 0.0, 0.0])
tau_ref = 36.0  # timer


def cstr_ode(t, x, tau=tau_ref):
    reaksjon = batch_ode(t, x)
    transport = (x_inn-x)/tau
    return ...

## Oppgave D1: Finn en stasjonær driftstilstand

Simuler lenge fra en rimelig starttilstand, og kontroller at

$$\|f(x^*)\|$$

er liten.

Rapporter:

- utløpskonsentrasjonene
- karbohydratomdanning
- hydrogenutstrømning $H/\tau$ i modellens enheter
- laktatoppbygging

In [ ]:
x0_cstr = np.array([80.0, 20.0, 10.0, 1.0, 1.0])

# Simuler for eksempel 1500 timer og finn x_stjerne.

## Oppgave D2: Oppholdstid

Undersøk

$$\tau=8,16,24,36,60,96\ \mathrm{timer}.$$

For hver verdi beregnes:

- karbohydratomdanning
- laktat i utløpet
- hydrogenrate ut av reaktoren
- hydrogen per innmatet karbohydrat

Diskuter avveiningen mellom høy gjennomstrømning og høy omdanning.

## D.2 To reaktorer i serie

En naturlig fordypning er:

- reaktor 1 optimalisert for rask karbohydratomdanning
- reaktor 2 med lengre oppholdstid for laktatomdanning

Utløpet fra reaktor 1 brukes som innløp til reaktor 2. Sammenlign med én reaktor med samme samlede oppholdstid.

# Del E: Linearisering og egenmoder

## E.1 Numerisk Jacobimatrise

Rundt en stasjonær CSTR-tilstand $x^*$ gjelder omtrent

$$
\boxed{\dot\xi=J\xi,\qquad\xi=x-x^*.}
$$

Jacobimatrisen beregnes med sentrale differanser:

$$
J_{ij}\approx
\frac{f_i(x^*+\varepsilon e_j)-f_i(x^*-\varepsilon e_j)}{2\varepsilon}.
$$

In [ ]:
def numerisk_jacobi(f, x, eps=1e-5):
    x = np.asarray(x, dtype=float)
    n = len(x)
    J = np.zeros((n, n))
    for j in range(n):
        e = np.zeros(n)
        e[j] = eps
        J[:, j] = ...
    return J

## Oppgave E1: Finn egenverdiene

Bruk likevekten fra del D. Beregn $J$, egenverdier og egenvektorer.

Tolk:

- raske negative egenverdier som rask karbohydrattilpasning
- langsommere egenverdier som laktat- og produktrespons
- fortynningsmoder knyttet til $1/\tau$

In [ ]:
# J = numerisk_jacobi(lambda t, x: cstr_ode(t, x, tau_ref), x_stjerne)
# egenverdier, P = np.linalg.eig(J)
# Sorter etter realdel og beregn tidskonstanter.

## E.2 Egenvektorer og biologisk tolkning

Normaliser egenvektorene og undersøk hvilke komponenter som dominerer hver mode.

Mulige tolkninger:

- karbohydratmode
- laktatmode
- produktutvaskingsmode
- hydrogenbokføringsmode

Merk at kumulative produkter i batch og konsentrasjoner i CSTR har ulik dynamisk betydning.

## Oppgave E2: Direkte og lineær respons

Øk innløpskarbohydratet med 2 prosent og sammenlign:

- den ikke-lineære CSTR-modellen
- den lineære modellen $\dot\xi=J\xi$
- en modal løsning dersom $J$ er diagonaliserbar

Gjenta med en større forstyrrelse og vurder når lineariseringen blir unøyaktig.

# Fordypning: Dynamiske biomasser

La $X_C$ og $X_L$ bli tilstander:

$$
\boxed{\dot X_C=Y_Cr_C-k_{d,C}X_C,}
$$

$$
\boxed{\dot X_L=Y_Lr_L-k_{d,L}X_L.}
$$

I en kontinuerlig reaktor må eventuell utvasking også tas med:

$$-\frac{X}{\tau}.$$

Dette kan gi:

- oppstartsperiode
- biomasseutvasking ved liten oppholdstid
- flere biologiske tidsskalaer
- endret stabilitet

Nye parametre må dokumenteres og følsomhetsanalyseres.

# Næringsstoffer og medium

Tilleggsdataene inneholder konsentrasjoner av blant annet nitrogen, fosfor, kalium, magnesium, kalsium og spormetaller i det syntetiske mediet.

I hovedmodellen antas disse å være tilstrekkelige og ikke-begrensende. Som fordypning kan én næringsfaktor $n(t)\in[0,1]$ multiplisere de biologiske ratene:

$$r_C\mapsto nr_C,\qquad r_L\mapsto nr_L.$$

Diskuter hvorfor en slik enkeltfaktor fortsatt er en grov representasjon av næringsbegrensning.

# Modellkritikk

Diskuter minst seks punkter:

- Utbyttekoeffisientene er på COD-basis.
- Effektive biomasser er konstante i hovedmodellen.
- Monod-lignende kinetikk beskriver ikke alle hemminger.
- pH er utelatt.
- Temperaturen er konstant.
- Hydrogenoverføring til gassfase er utelatt.
- Gassens partialtrykk påvirker ikke reaksjonene.
- Acetat og butyrat reagerer ikke videre.
- Ingen biomasseutvasking i hovedmodellen.
- Produktene antas fullstendig blandet.
- Reaktoren er idealisert som batch eller perfekt omrørt CSTR.
- Parameterne kan være korrelerte og usikre.
- Euler krever kontroll av tidssteget.
- Numerisk likevekt avhenger av simuleringstid og starttilstand.
- Lineariseringen gjelder bare nær valgt driftspunkt.

## Mulige videreføringer

- dynamiske biomasser
- temperaturavhengighet
- pH- og produkthemming
- gassfase og Henrys lov
- to reaktorer i serie
- parameterestimering fra måledata
- sammenligning med Gompertz-kurve
- optimal oppholdstid
- energibalanse for produsert hydrogen

# Oppsummering

Skriv en kort rapport der du forklarer

1. hvordan reaksjonsdiagrammet ga matrisen $N$,
2. hvordan to reaksjonstrinn ga samlet produktutbytte,
3. hvorfor COD-koeffisientene ikke er vanlige massefraksjoner,
4. hvordan metningskinetikken påvirket karbohydratforbruket,
5. hvorfor laktat først kunne bygge seg opp og senere avta,
6. hvordan matrisen gjorde den femkomponents ODE-en kompakt,
7. hvordan oppholdstid påvirket kontinuerlig drift,
8. hvilke biologiske tidsskalaer egenverdiene viste,
9. når den lineære modellen var en god tilnærming,
10. hvilke mekanismer som må legges til for en mer realistisk biohydrogenmodell.

## Faglig bakgrunn

Prosjektet bygger på en ADM1-inspirert biohydrogenmodell med karbohydrat- og laktatomdanning, oppgitte produktutbytter og metningskinetikk. Hovedmodellen er redusert slik at den støkiometriske matrisen, stoffbalansene og ODE-systemet forblir synlige.